In [34]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
import json
import scipy
np.float = float

from skmultiflow.drift_detection.adwin import ADWIN
from skmultiflow.drift_detection import DDM
from skmultiflow.drift_detection import EDDM

import sklearn
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import csr_matrix

import time
import random

In [2]:
reviews = pd.read_json("Amazon_Fashion.jsonl", lines=True)

In [4]:
reviews = reviews[reviews["rating"] != 3]
reviews = reviews[reviews["timestamp"] > "2015-01-01"]
reviews = reviews[reviews["timestamp"] < "2020-01-01"]

In [5]:
def create_matrix(df):
	
	N = len(df['user_id'].unique())
	M = len(df['product_id'].unique())
	
	# Map Ids to indices
	user_mapper = dict(zip(np.unique(df["user_id"]), list(range(N))))
	product_mapper = dict(zip(np.unique(df["product_id"]), list(range(M))))
	
	# Map indices to IDs
	user_inv_mapper = dict(zip(list(range(N)), np.unique(df["user_id"])))
	product_inv_mapper = dict(zip(list(range(M)), np.unique(df["product_id"])))
	
	user_index = [user_mapper[i] for i in df['user_id']]
	product_index = [product_mapper[i] for i in df['product_id']]

	X = csr_matrix((df["rating"], (product_index, user_index)), shape=(M, N))
	
	return X, user_mapper, product_mapper, user_inv_mapper, product_inv_mapper

In [6]:
reviews["product_id"] = reviews["parent_asin"]

In [7]:
X, user_mapper, product_mapper, user_inv_mapper, product_inv_mapper = create_matrix(reviews)

In [8]:
kNN = NearestNeighbors(n_neighbors=5, algorithm="brute", metric='cosine')
kNN.fit(X)

NearestNeighbors(algorithm='brute', metric='cosine')

In [9]:
def find_similar_products(product_id, X, k, metric='cosine', show_distance=False):

  neighbour_ids = []

  product_ind = product_mapper[product_id]
  product_vec = X[product_ind]
  k+=1
  product_vec = product_vec.reshape(1,-1)
  neighbour = kNN.kneighbors(product_vec, return_distance=show_distance)
  for i in range(0,k):
    n = neighbour.item(i)
    neighbour_ids.append(product_inv_mapper[n])
  neighbour_ids.pop(0)
  return neighbour_ids

In [14]:
product_list = list(product_inv_mapper.values())

In [ ]:
#test performance of ERB-Detectors

In [16]:
adwin = ADWIN()
entry_number = 0
entries = []

In [30]:
for i in range(100000):
    adwin.add_element(1)

In [31]:
time_list = []
for i in range(100):
    prod = random.sample(product_list,1)[0]
    time_start = time.perf_counter()
    find_similar_products(prod,X,4)
    time_elapsed = (time.perf_counter() - time_start)
    time_list.append(time_elapsed)
print(np.mean(time_list))
print(np.min(time_list))
print(np.max(time_list))

0.12900173599948175
0.10782010000548325
0.20501910001621582


In [32]:
time_list_with_drift = []
for i in range(100):
    prod = random.sample(product_list,1)[0]
    time_start = time.perf_counter()
    find_similar_products(prod,X,4)
    for j in range(len(recs)):
        adwin.add_element(j)
        if adwin.detected_change():
            entries.append(j)
    time_elapsed = (time.perf_counter() - time_start)
    time_list_with_drift.append(time_elapsed)
print(np.mean(time_list_with_drift))
print(np.min(time_list_with_drift))
print(np.max(time_list_with_drift))

0.13236059200222372
0.10689840000122786
0.19931610001367517


In [35]:
ddm = DDM(min_num_instances=30)
alerts_ddm = []
entry_number = 0
entries = []
for i in range(100000):
    ddm.add_element(0)

In [37]:
time_list_with_drift = []
for i in range(100):
    prod = random.sample(product_list,1)[0]
    time_start = time.perf_counter()
    find_similar_products(prod,X,4)
    for j in range(4):
        ddm.add_element(1)
        if ddm.detected_change():
            entries.append(1)
    time_elapsed = (time.perf_counter() - time_start)
    time_list_with_drift.append(time_elapsed)
print(np.mean(time_list_with_drift))
print(np.min(time_list_with_drift))
print(np.max(time_list_with_drift))

0.13570605600107227
0.10914230000344105
0.20428559998981655


In [41]:
ddm = EDDM()
alerts_ddm = []
entry_number = 0
entries = []
for i in range(100000):
    ddm.add_element(0)

In [42]:
time_list_with_drift = []
for i in range(100):
    prod = random.sample(product_list,1)[0]
    time_start = time.perf_counter()
    find_similar_products(prod,X,4)
    for j in range(4):
        ddm.add_element(1)
        if ddm.detected_change():
            entries.append(1)
    time_elapsed = (time.perf_counter() - time_start)
    time_list_with_drift.append(time_elapsed)
print(np.mean(time_list_with_drift))
print(np.min(time_list_with_drift))
print(np.max(time_list_with_drift))

0.1345572820000234
0.10802590000093915
0.20876760000828654
